In [1]:
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
import random
from tqdm.auto import tqdm
import pandas as pd
import itertools
from statistics import mode
import string

In [2]:
val_data = []
with open("dev.jsonl", "r") as f:
    data = f.readlines()

for line in data:
    val_data.append(json.loads(line))


test_data = []
with open("test.jsonl", "r") as f:
    data = f.readlines()

for line in data:
    test_data.append(json.loads(line))

In [3]:

BASE_MODEL_ID = "microsoft/phi-2"
# torch.set_default_device(device)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, trust_remote_code=True, device_map="auto")
tokenizer.pad_token = tokenizer.eos_token

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
print(  val_data[0].keys(), )
print(val_data[0]['question']['choices'], val_data[0]['question']['stem'])
# print(  test_data[0]['question']['choices'][3], )


# print(test_data[0].keys())


dict_keys(['id', 'question', 'answerKey', 'fact1', 'fact2', 'combinedfact', 'formatted_question'])
[{'text': 'sand', 'label': 'A', 'para': 'Generally if there is a beach on the shore, it is beautiful sand. What sand there is, is liberally peppered with seaweed. Faith is to the human what sand is to the ostrich. Sun, Sand and a perfect climate contribute to a lively youthful atmosphere. Simply described, a sand tray is a small sand box designed for indoor use. Blast Describes a shot from a sand bunker. Also the term sand is used interchangeably. Hookworms are often found in the soil or sand in moderate climates. What sand remains is but residue. Generally such soils are sands or loamy sands.'}, {'text': 'occurs over a wide range', 'label': 'B', 'para': "Engineering is a wide range of activities that can be described best in terms of functions. Climate A wide range of climatic conditions are present in the large geographical range of redbud. Aroids grow all over the world and occur in a 

In [5]:
from string import Template
prompt_template= Template('''Answer the following question using the context below by selecting the most likely option (A, B, C, D, E, F, G or H):\n
$question
context: $context      
                                                        
$options
$question
''')


# prompt_template2= Template('''Instruct: Answer the following question using the context provided, reason over it because only one of the context is relevant . Please generate only answer choice (1, 2, 3, 4, 5, 6, 7 or 8) without any explanations\n
# $question
# context: $context      
                                                        
# $options
# $question
# ''')


In [6]:
# Function to clean and split text into words
def preprocess(text):
    
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return set(text.split())


def choose_most_likely_option(options, candidate_answer):
    candidate_words = preprocess(candidate_answer)
    
    best_option = None
    max_overlap = 0
    
    for option in options:
        option_words = preprocess(option)
        overlap = len(candidate_words.intersection(option_words))
        
  
        if overlap > max_overlap:
            max_overlap = overlap
            best_option = option
            
    return best_option, options.index(best_option)+1

In [7]:
map_ans = {"A":1, "B":2, "C":3, "D":4, "E":5, "F":6, "G":7, "H":8}
def answer_reggexer(text,options, prompt_len):
    answer_only = text[prompt_len:]
    try:
        id  = answer_only.split(" correct option is ")[1][0]
        id = int(map_ans[id])
        


    except:
        try:
        # print(answer_only)
            _, id =  choose_most_likely_option(options, answer_only)
            # print(answer_only, id)
        except:


            id  = random.randint(1,8)
    return id
    

In [8]:

# data[0]
submission = {"answers":[]}
option_header = ["option A ", "option B ", "option C ", "option D ", "option E ","option F ", "option G ", "option H " ]

pbar = tqdm(range(len(val_data)))
k = 20
for example in val_data:
    options = []
    opts = []
    context = ''
    for i, sample in enumerate(example['question']['choices']):
        # print(key)
        
    
        options.append(option_header[i]+ sample['text'])
        opts.append((sample['text'], map_ans[option_header[i].split("option")[1][1]] ))
        context += '\n'+ sample['para']

    # combined_fact = example['combinedfact']
    combined_fact =example["fact1"] + '\n' + example["fact2"]


    all_permutations = list(itertools.permutations(opts))
    all_permutations = random.sample(all_permutations, k if len(all_permutations)>k else len(all_permutations))
    # print(all_permutations)
    batch_prompts = []
    batch_options = []
    option_maps = []
    for option_set in all_permutations:
        options  = []
        option_map = {}
        for i in range(len(option_header)):
            options.append(option_header[i] + option_set[i][0])
            option_map[i+1]  = option_set[i][1]
        option_maps.append(option_map)
        batch_options.append(options)
        batch_prompts.append(prompt_template.substitute( question  = example['question']['stem'], context=  combined_fact, options = "\n".join(options)))

    # prompt_sample = prompt_template.substitute( question  = example['question']['stem'], context=  combined_fact, options = "\n".join(options))

    # print(prompt_sample)

    input  = tokenizer(batch_prompts, padding="longest", return_tensors="pt").to("cuda")
    # print(input)
    out = model.generate(**input,  max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
    texts = tokenizer.batch_decode(out,skip_special_tokens=True)
    answers = []
    for text, options, option_map in zip(texts, batch_options, option_maps):
        
        id = answer_reggexer(text,options, len(batch_prompts[0]))
        answers.append(option_map[id])

    # print(mode(answers), answers)


    submission["answers"].append(mode(answers))
    pd.DataFrame(submission).to_csv("phik"+str(k)+"_qasc_f1f2.csv")
    
    pbar.set_description(f"Pred: {id:.4f}")
    pbar.update(1)
pbar.close()
# print(answer_only)

  0%|          | 0/926 [00:00<?, ?it/s]

In [9]:
print(combined_fact)

Renal failure may be treated with dialysis


In [10]:
print(batch_prompts[0], texts[0])

Answer the following question using the context below by selecting the most likely option (A, B, C, D, E, F, G or H):

What may renal failure be treated with?
context: Renal failure may be treated with dialysis      
                                                        
option A Protein
option B saves lives
option C Lymphocytes
option D ibuprofen
option E Lymph fluid
option F dandelions
option G dialysis
option H Laboratory
What may renal failure be treated with?
 Answer the following question using the context below by selecting the most likely option (A, B, C, D, E, F, G or H):

What may renal failure be treated with?
context: Renal failure may be treated with dialysis      
                                                        
option A Protein
option B saves lives
option C Lymphocytes
option D ibuprofen
option E Lymph fluid
option F dandelions
option G dialysis
option H Laboratory
What may renal failure be treated with?

Solution 0:

The correct option is G dialysis.

Renal fa

In [11]:
print(batch_prompts[1])


Answer the following question using the context below by selecting the most likely option (A, B, C, D, E, F, G or H):

What may renal failure be treated with?
context: Renal failure may be treated with dialysis      
                                                        
option A Laboratory
option B dialysis
option C saves lives
option D dandelions
option E ibuprofen
option F Lymphocytes
option G Lymph fluid
option H Protein
What may renal failure be treated with?



In [12]:
print(text[0])

A


In [13]:
print(text[1])


n


In [14]:
opts

[('Laboratory', 1),
 ('Lymphocytes', 2),
 ('saves lives', 3),
 ('dialysis', 4),
 ('Lymph fluid', 5),
 ('dandelions', 6),
 ('ibuprofen', 7),
 ('Protein', 8)]

In [15]:
submission["answers"]


[6,
 7,
 4,
 6,
 2,
 4,
 6,
 1,
 6,
 8,
 1,
 2,
 1,
 3,
 7,
 4,
 4,
 4,
 2,
 8,
 7,
 1,
 5,
 3,
 1,
 3,
 5,
 3,
 6,
 4,
 6,
 3,
 5,
 1,
 2,
 8,
 6,
 6,
 5,
 2,
 4,
 2,
 2,
 4,
 6,
 8,
 6,
 5,
 5,
 3,
 7,
 2,
 1,
 5,
 8,
 3,
 1,
 7,
 7,
 3,
 8,
 3,
 2,
 5,
 3,
 6,
 6,
 3,
 7,
 3,
 7,
 3,
 8,
 5,
 3,
 5,
 6,
 3,
 3,
 6,
 1,
 5,
 1,
 6,
 2,
 5,
 7,
 8,
 7,
 2,
 7,
 2,
 4,
 6,
 1,
 4,
 8,
 1,
 4,
 4,
 1,
 8,
 1,
 6,
 6,
 8,
 5,
 3,
 8,
 2,
 5,
 8,
 1,
 5,
 4,
 2,
 6,
 1,
 5,
 4,
 3,
 3,
 3,
 8,
 8,
 3,
 1,
 4,
 7,
 3,
 3,
 4,
 6,
 2,
 2,
 1,
 7,
 4,
 8,
 7,
 5,
 5,
 2,
 2,
 5,
 8,
 1,
 4,
 6,
 2,
 1,
 4,
 4,
 2,
 4,
 4,
 8,
 1,
 1,
 1,
 8,
 5,
 5,
 5,
 3,
 2,
 1,
 6,
 7,
 7,
 6,
 6,
 8,
 7,
 1,
 1,
 6,
 8,
 5,
 4,
 2,
 2,
 5,
 2,
 1,
 4,
 4,
 8,
 7,
 3,
 3,
 8,
 6,
 3,
 1,
 4,
 4,
 6,
 2,
 6,
 1,
 1,
 8,
 4,
 3,
 1,
 7,
 7,
 6,
 1,
 8,
 8,
 2,
 8,
 6,
 6,
 6,
 2,
 7,
 1,
 3,
 7,
 3,
 5,
 7,
 1,
 8,
 5,
 5,
 2,
 3,
 5,
 7,
 7,
 7,
 4,
 2,
 4,
 8,
 3,
 3,
 4,
 7,
 4,
 3,
 5,
 3,
 2,
 4,
 2,


In [16]:
tokenizer.batch_decode(out)[0]

'Answer the following question using the context below by selecting the most likely option (A, B, C, D, E, F, G or H):\n\nWhat may renal failure be treated with?\ncontext: Renal failure may be treated with dialysis      \n                                                        \noption A Protein\noption B saves lives\noption C Lymphocytes\noption D ibuprofen\noption E Lymph fluid\noption F dandelions\noption G dialysis\noption H Laboratory\nWhat may renal failure be treated with?\n\nSolution 0:\n\nThe correct option is G dialysis.\n\nRenal failure may be treated with dialysis. Dialysis is a process that filters the blood and removes excess fluid and waste products when the kidneys are not working properly.'

In [17]:
context

"\nRoutine renal laboratory data have been compared with histopathological findings. Company sells dialysis products and list laboratory tests and renal services. Laboratory quantities of oxidizers can be treated. Failure to participate in the laboratory part of the course automatically results in course failure. Laboratory studies reveal a normal hemoglobin and hematocrit and renal panel. Failure to complete the laboratory work is grounds for failure in the course. And what a laboratory it is. Cardiac, pulmonary and renal laboratories begin to function. Field failures are compared with failures in the laboratory. Laboratory findings are also variable, reflecting the primary disease, GN or renal failure.\nWhat role, if any, does BLyS have in regulating B-lymphocyte function. Clemson University researchers have discovered a new way to treat chronic lymphocytic leukemia. Poor B lymphocyte function can recover in patients treated for lead poisoning. T lymphocytes mediate leaflet destructi